# Mineração de Dados — Atividade prática de Data Profiling

**Universidade Federal do Ceará · Departamento de Computação**
Disciplina de Mineração de Dados — Prof. José Macedo

---

**Dupla:** Samyra Almeida; Yasmin Santos de Amorim - 566326
**Data:** _(dd/mm/aaaa)_

---

## O que fazer

Aplicar o `ydata-profiling` a **cinco conjuntos do scikit-learn** e transformar o
relatório automático em **decisões de análise**.

O HTML gerado pela ferramenta é **insumo**, não é o relatório. O que vale é a
interpretação que vocês escrevem nas células de markdown deste notebook.

## O que entregar

- este notebook, executando do zero sem erros (`Kernel → Restart & Run All`);
- os cinco arquivos `perfil_*.html` gerados;
- o mini-relatório em PDF, de 4 a 6 páginas;
- tudo em um ZIP ou repositório Git.

## Como este notebook está organizado

| Parte | Conteúdo |
|-------|----------|
| 0 | Preparação do ambiente |
| 1 | Carga dos 5 datasets e ficha técnica comparativa |
| 2 | Um perfil por dataset + roteiro de 6 passos + achados |
| 3 | Análise cruzada entre os cinco |
| 4 | Recomendações de preparação |
| 5 | Limites do profiling automático |
| 6 | Checklist de entrega |

---
## Parte 0 — Preparação do ambiente

In [ ]:
# Rode uma única vez. No Colab, reinicie o runtime depois da instalação.
%pip install -q "ydata-profiling[notebook]" scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import ydata_profiling
from ydata_profiling import ProfileReport

# Registrar as versões é parte da reprodutibilidade — copie isto para a capa do relatório.
print("python          ", sys.version.split()[0])
print("numpy           ", np.__version__)
print("pandas          ", pd.__version__)
print("scikit-learn    ", sklearn.__version__)
print("ydata-profiling ", ydata_profiling.__version__)

python           3.13.15
numpy            2.1.3
pandas           2.2.3
scikit-learn     1.6.1
ydata-profiling  4.18.4


/tmp/ipykernel_3176/890274313.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  import ydata_profiling


---
## Parte 1 — Carga dos 5 datasets

Todos vêm de `sklearn.datasets`. Apenas o **California Housing** faz download na
primeira execução (`fetch_`); os outros quatro já acompanham a biblioteca.

`as_frame=True` devolve um `Bunch` cujo atributo `.frame` já traz atributos e alvo
em um único `DataFrame`, e cujo `.DESCR` traz o dicionário de dados.

In [ ]:
from sklearn.datasets import (
    load_iris,
    load_wine,
    load_breast_cancer,
    load_diabetes,
    fetch_california_housing,
)

LOADERS = {
    "iris":          load_iris,
    "wine":          load_wine,
    "breast_cancer": load_breast_cancer,
    "diabetes":      load_diabetes,
    "california":    fetch_california_housing,
}

bunches = {nome: loader(as_frame=True) for nome, loader in LOADERS.items()}
dfs = {nome: b.frame.copy() for nome, b in bunches.items()}

for nome, df in dfs.items():
    print(f"{nome:<15} {df.shape[0]:>6} linhas x {df.shape[1]:>3} colunas")

iris               150 linhas x   5 colunas
wine               178 linhas x  14 colunas
breast_cancer      569 linhas x  31 colunas
diabetes           442 linhas x  11 colunas
california       20640 linhas x   9 colunas


### Leia o dicionário de dados

Não pule esta etapa: o passo 1 do roteiro (**Contexto**) depende dela. Troque o nome
para inspecionar cada conjunto.

In [ ]:
print(bunches["california"].DESCR[:2000])

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

### Ficha técnica comparativa

Esta tabela é o **item 2 do mini-relatório**. Ela responde, de uma só vez, ao passo
"Estrutura" e a boa parte do passo "Qualidade" dos cinco conjuntos.

In [ ]:
def ficha_tecnica(df: pd.DataFrame) -> dict:
    """Resumo estrutural e de qualidade de um DataFrame."""
    n_linhas, n_colunas = df.shape
    numericas = df.select_dtypes(include="number").columns
    constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    celulas = n_linhas * n_colunas
    return {
        "linhas": n_linhas,
        "colunas": n_colunas,
        "numericas": len(numericas),
        "nao_numericas": n_colunas - len(numericas),
        "% ausentes": round(100 * df.isna().sum().sum() / celulas, 2),
        "duplicatas": int(df.duplicated().sum()),
        "col. constantes": len(constantes),
        "% zeros": round(100 * (df[numericas] == 0).sum().sum()
                         / max(n_linhas * len(numericas), 1), 2),
        "memoria (KB)": round(df.memory_usage(deep=True).sum() / 1024, 1),
    }


ficha = pd.DataFrame({nome: ficha_tecnica(df) for nome, df in dfs.items()}).T
ficha

,linhas,colunas,numericas,nao_numericas,% ausentes,duplicatas,col. constantes,% zeros,memoria (KB)
iris,150.0,5.0,5.0,0.0,0.0,1.0,0.0,6.67,6.0
wine,178.0,14.0,14.0,0.0,0.0,0.0,0.0,2.37,19.6
breast_cancer,569.0,31.0,31.0,0.0,0.0,0.0,0.0,1.64,137.9
diabetes,442.0,11.0,11.0,0.0,0.0,0.0,0.0,0.00,38.1
california,20640.0,9.0,9.0,0.0,0.0,0.0,0.0,0.00,1451.4


**Pergunta para o relatório:** olhando só para esta tabela, qual dos cinco
conjuntos parece o mais "sujo"? Sua resposta muda depois de abrir os perfis
completos? Por quê?

---
## Parte 2 — Um perfil por dataset

A função abaixo gera e salva o relatório. Os argumentos seguem as recomendações
vistas em aula:

- `minimal=True` para conjuntos com muitas colunas (desliga correlações e
  interações caras);
- amostragem para conjuntos com muitas linhas.

> **Atenção:** `minimal=True` e amostragem aceleram o perfil, mas escondem
> justamente as correlações que interessam ao relatório. Use como primeira
> passada e **confirme no conjunto completo** antes de escrever os achados.

In [ ]:
def gerar_perfil(nome: str, df: pd.DataFrame, minimal: bool = False,
                 amostra: int | None = None, salvar: bool = True) -> ProfileReport:
    """Gera o ProfileReport de um dataset e salva em perfil_<nome>.html."""
    dados = df
    titulo = f"Perfil — {nome}"
    if amostra is not None and len(df) > amostra:
        dados = df.sample(amostra, random_state=42)
        titulo += f" (amostra de {amostra} linhas)"

    perfil = ProfileReport(dados, title=titulo, explorative=not minimal,
                           minimal=minimal)
    if salvar:
        perfil.to_file(f"perfil_{nome}.html")
        print(f"salvo: perfil_{nome}.html")
    return perfil


perfis = {}

### 2.1 — Iris

*Linha de base limpa — mas confira as duplicatas.*

**Roteiro dos 6 passos** — responda cada item em uma frase, com o número que sustenta a resposta.

1. **Contexto** — o que o `DESCR` diz sobre origem, unidades e significado das variáveis?
2. **Estrutura** — quantas linhas e colunas? Os tipos inferidos batem com os tipos semânticos?
3. **Qualidade** — ausentes, duplicatas, colunas constantes, cardinalidade, excesso de zeros?
4. **Distribuições** — assimetria, curtose, multimodalidade, outliers pelo IQR?
5. **Relações** — quais correlações são fortes? Há redundância ou risco de vazamento do alvo?
6. **Decisão** — o que isso muda no pré-processamento e na escolha do algoritmo?

In [ ]:
perfis["iris"] = gerar_perfil("iris", dfs["iris"])
perfis["iris"].to_notebook_iframe()

In [ ]:
# Pista: o relatório aponta linhas repetidas. Quais são?
dfs["iris"][dfs["iris"].duplicated(keep=False)]

### Achados — Iris

> Preencha três descobertas. Cada uma precisa das **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 | Quanto maior o comprimento da pétala maior tende a ser sua largura | Na seção "Correlations", a correlação entre petal length e petal petal width é 0,938  |                      |
| 2 |  Existe um registro duplicado                   |  Na seção "Duplicate rows", é mostrada uma linha que aparece duas vezes com exatamente os mesmos valores: (sepal length=5.8, sepal width=2.7, petal length=5.1, petal width=1.9, target=2, # duplicates=2)                                     |                      |
| 3 |                     |                                        |                      |

**Respostas do roteiro (6 passos):**

1. Contexto: O dataset Iris possui 150 observações de 3 espécies (setosa, versicolor e virginica). Ele é baseado em quatro medidas físicas, o comprimento e largura da sépala e da pétala de cada flor, em centímetros.   
2. Estrutura: 150 linhas x 5 colunas, 4 variáveis numéricas e 1 categórica. A variável categórica é a target e ela possui 3 categorias, com 50 observações em cada categoria.
3. Qualidade: Nenhum valor ausente, a seção de variáveis aponta Missing 0%, Zeros 0% e Infinite 0% em todas as colunas, mas há 1 linha duplicada.
4. Distribuições:
5. Relações: Com 0,938, petal lenght e petal width têm correlação muito forte entre si e também são as variáveis mais correlacionadas com a classe target com 0,890 e 0,924, isso as torna excelentes preditoras. Não há indício de vazamentos de dados, pois são características biológicas reais
6. Decisão:

### 2.2 — Wine

*Escalas muito heterogêneas entre as 13 variáveis.*

**Roteiro dos 6 passos** — responda cada item em uma frase, com o número que sustenta a resposta.

1. **Contexto** — o que o `DESCR` diz sobre origem, unidades e significado das variáveis?
2. **Estrutura** — quantas linhas e colunas? Os tipos inferidos batem com os tipos semânticos?
3. **Qualidade** — ausentes, duplicatas, colunas constantes, cardinalidade, excesso de zeros?
4. **Distribuições** — assimetria, curtose, multimodalidade, outliers pelo IQR?
5. **Relações** — quais correlações são fortes? Há redundância ou risco de vazamento do alvo?
6. **Decisão** — o que isso muda no pré-processamento e na escolha do algoritmo?

In [ ]:
perfis["wine"] = gerar_perfil("wine", dfs["wine"])
perfis["wine"].to_notebook_iframe()

In [ ]:
# Pista: compare as amplitudes. Qual variável dominaria uma distância euclidiana?
dfs["wine"].describe().T[["min", "max", "mean", "std"]].sort_values("std")

,min,max,mean,std
nonflavanoid_phenols,0.13,0.66,0.361854,0.124453
hue,0.48,1.71,0.957449,0.228572
ash,1.36,3.23,2.366517,0.274344
proanthocyanins,0.41,3.58,1.590899,0.572359
total_phenols,0.98,3.88,2.295112,0.625851
od280/od315_of_diluted_wines,1.27,4.00,2.611685,0.709990
target,0.00,2.00,0.938202,0.775035
alcohol,11.03,14.83,13.000618,0.811827
flavanoids,0.34,5.08,2.029270,0.998859
malic_acid,0.74,5.80,2.336348,1.117146


### Achados — Wine

> Preencha três descobertas. Cada uma precisa das **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 |As variáveis possuem escalas muito diferentes.      |  O proline varia em centenas  (278–1680, SD=315) enquanto  nonflavanoid_phenols (0,13–0,66, SD=0,12)  varia em décimos. Informações presentes em  describe().T                                      |  Padronizar (StandardScaler) antes de qualquer método baseado em distância (KNN, K-means, PCA)                    |
| 2 |    Algumas variáveis estão repetindo a mesma informação                 |     A aba "Correlations" mostra uma correlação muito alta (0,879) entre as colunas flavanoids e total_phenols.                                   |   Considerar remoção de uma dessas colunas repetidas antes de treinar o modelo para que ele não processe informações duplicadas.                   |
| 3 | Algumas variáveis são boas para identificar o produtor do vinho.                    | Na aba "Correlations", as colunas flavanoids (r=0,752), color_intensity (r=0,649) e proline (r=0,644) mostram as maiores correlações com o target.                 |    Na hora de montar o modelo, devemos priorizar essas colunas, pois elas são as que mais ajudam a separar as 3 classes.                  |

**Respostas do roteiro (6 passos):**

1. Contexto: O dataset Wine possui 178 observações de 3 produtores diferentes na mesma região da Itália. Ele contém os resultados de análises químicas de 13 atributos, com o objetivo de classificar a origem do vinho com base nessas medições.
2. Estrutura: 178 linhas x 14 colunas, 13 variáveis númericas e 1 categórica. As classes estão levemente desbalanceadas com a seguintes distribuição, class_0: 59, class_1: 71 e class_2: 48.
3. Qualidade: Não há valores ausentes, linhas duplcadas, valores infinitos ou colunas constantes. O dataset está estruturalmente limpo
4. Distribuições: As variáveis possuem escalas de grandeza bastante diferentes, com a proline apresentando a maior amplitude (278 a 1680) e o maior desvio padrão (314,91). Isso significa que, se usarmos o dataset do jeito que está, a proline vai dominar e distorcer totalmente os cálculos de distância em modelos como KNN, K-means ou PCA.
5. Relações: Há redundância entre total_phenols e flavanoids (0,879), o que indica que ambas as variáveis medem características muito semelhantes do vinho e fornecem informações repetidas. O target possui forte correlação com variáveis como flavanoids (0,752) e proline (0,644), indicando que essas substâncias são fundamentais para distinguir um produtor do outro. Por outro lado, essa mesma redundância entre preditores é um risco de multicolinearidade para modelos lineares, o que reforça a necessidade de PCA ou seleção de variáveis antes de treinar um classificador linear.
6. Decisão: Padronizar os dados é obrigatório antes de usar KNN, K-means, SVM (RBF) ou PCA.

### 2.3 — Breast Cancer

*Redundância: mean, error e worst do mesmo atributo.*

**Roteiro dos 6 passos** — responda cada item em uma frase, com o número que sustenta a resposta.

1. **Contexto** — o que o `DESCR` diz sobre origem, unidades e significado das variáveis?
2. **Estrutura** — quantas linhas e colunas? Os tipos inferidos batem com os tipos semânticos?
3. **Qualidade** — ausentes, duplicatas, colunas constantes, cardinalidade, excesso de zeros?
4. **Distribuições** — assimetria, curtose, multimodalidade, outliers pelo IQR?
5. **Relações** — quais correlações são fortes? Há redundância ou risco de vazamento do alvo?
6. **Decisão** — o que isso muda no pré-processamento e na escolha do algoritmo?

In [ ]:
# 30 colunas: comece pelo modo enxuto e depois rode sem "minimal" para ver as correlações.
perfis["breast_cancer"] = gerar_perfil("breast_cancer", dfs["breast_cancer"])
perfis["breast_cancer"].to_notebook_iframe()

In [ ]:
# Pista: quantos pares de atributos passam de 0,95 de correlação absoluta?
X = dfs["breast_cancer"].drop(columns="target")
corr = X.corr().abs()
pares = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack()
             .sort_values(ascending=False))
print("pares com |r| > 0,95:", int((pares > 0.95).sum()))
pares.head(10)

pares com |r| > 0,95: 15


,,0
mean radius,mean perimeter,0.997855
worst radius,worst perimeter,0.993708
mean radius,mean area,0.987357
mean perimeter,mean area,0.986507
worst radius,worst area,0.984015
worst perimeter,worst area,0.977578
radius error,perimeter error,0.972794
mean perimeter,worst perimeter,0.970387
mean radius,worst radius,0.969539
mean perimeter,worst radius,0.969476


### Achados — Breast Cancer

> Preencha três descobertas. Cada uma precisa das **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 | Tem muita coluna repetindo a mesma informação da célula.                    |   O modelo listou 15 pares de colunas com correlação acima de 0,95. O par mean radius e mean perimeter, por exemplo, bateu 0,997.                                     |  Será preciso remover algumas colunas ou usar PCA para evitar problemas causados por variáveis muito semelhantes                    |
| 2 | Os valores das colunas estão em grandezas diferentes.                    |    No resumo estatístico, a coluna mean area chega a 2501.0, enquanto a mean smoothness não passa de 0.163.                                    | É necessário padronizar os dados com StandardScaler. Sem isso, no KNN, a variável area terá mais influência nos cálculos apenas por apresentar valores maiores.                     |
| 3 | Seis colunas de concavidade repetem exatamente os mesmos 13 zeros.                    |  As colunas mean concavity, mean concave points, concavity error, concave points error, worst concavity e worst concave points marcam Zeros: 13 (2,3%)".                                 |   Esses zeros parecem representar uma característica real dos dados, então é importante verificar se estão concentrados na classe benigna antes de tratá-los como outliers                   |

**Respostas do roteiro (6 passos):**

1. Contexto: O dataset Breast Cancer é uma base de dados médica usada para diagnóstico de câncer de mama. Ela traz várias medidas geométricas das células (como raio, textura e área) para que o modelo consiga classificar se o tumor do paciente é maligno ou benigno.
2. Estrutura: 569 linhas × 31 colunas, 30 variáveis numéricas contínuas e 1 categórica (target, o diagnóstico binário).
3. Qualidade: Não tem nenhum dado faltando (0% missing) e nenhuma linha duplicada.
4. Distribuições: As variáveis têm escalas muito diferentes. Por exemplo, a média de area varia de 143,5 a 2501, enquanto smoothness varia apenas de 0,053 a 0,163.
5. Relações: Tem muita coluna repetindo a mesma informação. O relatório aponta 15 pares de variáveis com correlação alta (passando de 0,95). O motivo disso é que a base cria 3 colunas para a mesma medida (a média, o erro e o pior caso). Além disso, medidas geométricas como raio e perímetro acabam entregando a mesma matemática para o modelo.
6. Decisão: A padronização é obrigatória por causa das escalas heterogêneas. Também é preciso usar PCA ou fazer uma seleção manual para retirar as colunas redundantes; caso contrário, os modelos podem se perder.

### 2.4 — Diabetes

*Já vem centrado e escalonado; `sex` é categórica disfarçada de número.*

**Roteiro dos 6 passos** — responda cada item em uma frase, com o número que sustenta a resposta.

1. **Contexto** — o que o `DESCR` diz sobre origem, unidades e significado das variáveis?
2. **Estrutura** — quantas linhas e colunas? Os tipos inferidos batem com os tipos semânticos?
3. **Qualidade** — ausentes, duplicatas, colunas constantes, cardinalidade, excesso de zeros?
4. **Distribuições** — assimetria, curtose, multimodalidade, outliers pelo IQR?
5. **Relações** — quais correlações são fortes? Há redundância ou risco de vazamento do alvo?
6. **Decisão** — o que isso muda no pré-processamento e na escolha do algoritmo?

In [ ]:
perfis["diabetes"] = gerar_perfil("diabetes", dfs["diabetes"])
perfis["diabetes"].to_notebook_iframe()

In [ ]:
# Pista: os atributos já vêm centrados e escalonados? E quantos valores tem "sex"?
resumo = dfs["diabetes"].describe().T[["mean", "std", "min", "max"]]
resumo["n_unicos"] = dfs["diabetes"].nunique()
resumo

,mean,std,min,max,n_unicos
age,-2.511817e-19,0.047619,-0.107226,0.110727,58
sex,1.230790e-17,0.047619,-0.044642,0.050680,2
bmi,-2.245564e-16,0.047619,-0.090275,0.170555,163
bp,-4.797570e-17,0.047619,-0.112399,0.132044,100
s1,-1.381499e-17,0.047619,-0.126781,0.153914,141
s2,3.918434e-17,0.047619,-0.115613,0.198788,302
s3,-5.777179e-18,0.047619,-0.102307,0.181179,63
s4,-9.042540e-18,0.047619,-0.076395,0.185234,66
s5,9.293722e-17,0.047619,-0.126097,0.133597,184
s6,1.130318e-17,0.047619,-0.137767,0.135612,56


### Achados — Diabetes

> Preencha três descobertas. Cada uma precisa das **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 | O dataset já está todo centralizado e padronizado.                    |       Todas as variáveis preditoras têm média zero e desvio padrão idêntico de 0,0476.                                 |     Os dados já foram padronizados, então não é preciso usar o StandardScaler                 |
| 2 |  Forte correlação entre alguns exames de colesterol do sangue                   |     A matriz de correlação aponta uma ligação muito forte de 0,879 entre s1 e s2                                    |  Tem uma  uma repetição  de dados entre os exames. Vale avaliar se remove uma delas ou se usa PCA para simplificar o modelo linear.                    |
| 3 | A variável sex, apesar de numérica, é na verdade uma categoria binária                    |  A seção Variables classifica sex como "Categorical" com apenas 2 valores distintos (-0,0446 e 0,0507).                                      |      Deve ser tratada como uma variável binária, e não como uma variável contínua. Como ela já possui apenas dois valores, pode ser mantida como está ou codificada como categórica                |

**Respostas do roteiro (6 passos):**

1. Contexto: O dataset Diabetes é uma base de dados médica com 442 pacientes diagnosticados com diabetes. O objetivo é prever uma medida quantitativa da progressão da doença após um ano, usando exames de sangue e características básicas.
2. Estrutura: 442 linhas x 11 colunas, 10 variáveis numéricas e 1 categórica.
3. Qualidade: Tem zero dados ausentes e nenhuma linha duplicada.
4. Distribuições: Nesse dataset, as variáveis já vêm todas com média zero e o mesmo desvio padrão (0,0476), ou seja, elas já foram normalizadas. O target é uma pontuação contínua que vai de 25 a 346
5. Relações: As maiores correlações aparecem entre as variáveis de colesterol: s1xs2 = 0,879 e s3xs4 = -0,790. Em target, s5 (r=0,589) e bmi (r=0,561) são as variáveis mais preditivas, sex tem correlação exatamente 0,000 com o target
6. Decisão: Sex deve ser tratada como variável binária, e podemos reduzir a redundância entre s1/s2 e s3/s4. Como o target é contínuo, devemos usar técnicas de regressão

### 2.5 — California Housing

*Outliers extremos e alvo truncado no teto da escala.*

**Roteiro dos 6 passos** — responda cada item em uma frase, com o número que sustenta a resposta.

1. **Contexto** — o que o `DESCR` diz sobre origem, unidades e significado das variáveis?
2. **Estrutura** — quantas linhas e colunas? Os tipos inferidos batem com os tipos semânticos?
3. **Qualidade** — ausentes, duplicatas, colunas constantes, cardinalidade, excesso de zeros?
4. **Distribuições** — assimetria, curtose, multimodalidade, outliers pelo IQR?
5. **Relações** — quais correlações são fortes? Há redundância ou risco de vazamento do alvo?
6. **Decisão** — o que isso muda no pré-processamento e na escolha do algoritmo?

In [ ]:
# 20.640 linhas: a amostra dá o perfil rápido; confirme os outliers no conjunto completo.
perfis["california"] = gerar_perfil("california", dfs["california"], amostra=5000)
perfis["california"].to_notebook_iframe()

In [ ]:
# Pista: olhe os máximos e o topo da escala do alvo.
print(dfs["california"][["AveRooms", "AveOccup", "MedHouseVal"]].describe().T)
print()
teto = dfs["california"]["MedHouseVal"].max()
n_teto = (dfs["california"]["MedHouseVal"] >= teto).sum()
print(f"valor máximo do alvo: {teto}")
print(f"linhas exatamente no máximo: {n_teto} "
      f"({100 * n_teto / len(dfs['california']):.1f}% do total)")

               count      mean        std       min       25%       50%  \
AveRooms     20640.0  5.429000   2.474173  0.846154  4.440716  5.229129   
AveOccup     20640.0  3.070655  10.386050  0.692308  2.429741  2.818116   
MedHouseVal  20640.0  2.068558   1.153956  0.149990  1.196000  1.797000   

                  75%          max  
AveRooms     6.052381   141.909091  
AveOccup     3.282261  1243.333333  
MedHouseVal  2.647250     5.000010  

valor máximo do alvo: 5.00001
linhas exatamente no máximo: 965 (4.7% do total)


### Achados — California Housing

> Preencha três descobertas. Cada uma precisa das **três partes**: observação, evidência numérica e consequência prática.

| # | O que foi observado | Evidência (valor / seção do relatório) | Consequência prática |
|---|---------------------|----------------------------------------|----------------------|
| 1 |                     |                                        |                      |
| 2 |                     |                                        |                      |
| 3 |                     |                                        |                      |

**Respostas do roteiro (6 passos):**

1. Contexto: O dataset California Housing é uma base de dados macroeconômica baseada no censo dos Estados Unidos de 1990 para a Califórnia. O objetivo é prever o valor mediano das residências em cada setor censitário.
2. Estrutura: 20.640 linhas x 9 colunas, sendo todas númericas
3. Qualidade:
4. Distribuições:
5. Relações:
6. Decisão:

---
## Parte 3 — Análise cruzada

Agora compare os cinco. Esta é a parte que separa um relatório descritivo de um
relatório analítico.

In [ ]:
# Um quadro comparativo para apoiar a discussão. Acrescente as colunas que julgar úteis.
resumo_alvo = {}
for nome, df in dfs.items():
    alvo = df["target"] if "target" in df else df["MedHouseVal"]
    if alvo.nunique() <= 20:
        contagem = alvo.value_counts().sort_index()
        desc = " / ".join(str(v) for v in contagem.values)
        tipo = f"{alvo.nunique()} classes"
        balanco = round(contagem.min() / contagem.max(), 2)
    else:
        desc = f"{alvo.min():.2f} a {alvo.max():.2f}"
        tipo = "contínuo"
        balanco = np.nan
    resumo_alvo[nome] = {"tipo do alvo": tipo, "distribuição": desc,
                         "balanceamento": balanco}

pd.concat([ficha, pd.DataFrame(resumo_alvo).T], axis=1)

**Escreva aqui a análise cruzada** (item 4 do mini-relatório, ~1 página):

- Que problema de qualidade aparece em **mais de um** dos cinco conjuntos?
- Qual deles exige **mais** pré-processamento antes de qualquer modelagem? E qual exige menos?
- Nenhum dos cinco tem valores ausentes. Isso é bom sinal ou é artefato de serem
  datasets didáticos, já curados? O que muda em dados reais?
- O balanceamento das classes influencia a escolha da métrica de avaliação. Como?

_(sua resposta)_

---
## Parte 4 — Recomendações de preparação

Para cada dataset, proponha um pipeline curto e **justifique cada etapa com um
achado do profiling**. Uma etapa sem evidência é chute.

In [ ]:
# Esqueleto para você adaptar. Não precisa treinar modelo: o foco é a preparação.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

exemplo = Pipeline([
    ("escala", StandardScaler()),   # justificativa: <achado do profiling>
])
exemplo

| Dataset | Etapa proposta | Achado que justifica |
|---------|----------------|----------------------|
| Iris | | |
| Wine | | |
| Breast Cancer | | |
| Diabetes | | |
| California Housing | | |

---
## Parte 5 — Limites do profiling automático

O `ydata-profiling` calcula muita coisa, mas não sabe nada sobre o problema de
negócio. Aponte pelo menos **três** limitações que vocês encontraram na prática.
Alguns fios para puxar:

- A ferramenta consegue dizer se um valor extremo é **erro de medição** ou
  **fenômeno real**?
- Ela reconhece que uma coluna numérica é, na verdade, **categórica** ou um
  **identificador**?
- Ela detecta violação de **regra de negócio** (por exemplo, uma data de saída
  anterior à de entrada)?
- Correlação alta entre um atributo e o alvo é sinal de bom preditor ou de
  **vazamento**? Quem decide?

_(sua resposta)_

---
## Parte 6 — Checklist de entrega

Rode a célula abaixo antes de fechar o notebook: ela confere se os cinco HTMLs
foram mesmo gerados.

In [ ]:
from pathlib import Path

esperados = [f"perfil_{nome}.html" for nome in LOADERS]
for arquivo in esperados:
    caminho = Path(arquivo)
    if caminho.exists():
        print(f"[ok]    {arquivo:<28} {caminho.stat().st_size / 1024:>8.0f} KB")
    else:
        print(f"[FALTA] {arquivo}")

**Antes de enviar, confirme:**

- [ ] `Kernel → Restart & Run All` executa do início ao fim sem erro
- [ ] os cinco `perfil_*.html` estão na pasta
- [ ] cada dataset tem 3 achados preenchidos, com evidência e consequência
- [ ] a análise cruzada responde às quatro perguntas da Parte 3
- [ ] o `relatorio.pdf` tem entre 4 e 6 páginas
- [ ] as versões das bibliotecas estão na capa do relatório

---

**Como será avaliado**

| Critério | Peso |
|----------|------|
| Interpretação: achado → implicação | 30% |
| Corretude das estatísticas relatadas | 20% |
| Análise comparativa entre os 5 datasets | 20% |
| Reprodutibilidade do notebook | 20% |
| Clareza e concisão da escrita | 10% |